In [4]:
path = '../data/weather_data.json'
import pandas as pd
import json
from pathlib import Path

In [2]:



with open(path) as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head()

,location.name,location.region,location.country,location.lat,location.lon,location.tz_id,location.localtime_epoch,location.localtime,current.last_updated_epoch,current.last_updated,...,current.air_quality.o3,current.air_quality.so2,current.air_quality.pm2_5,current.air_quality.pm10,current.air_quality.us-epa-index,current.air_quality.gb-defra-index,current.short_rad,current.diff_rad,current.dni,current.gti
0,Sao Paulo,Sao Paulo,Brazil,-23.5333,-46.6167,America/Sao_Paulo,1793190519,2026-10-28 09:28,1793189700,2026-10-28 09:15,...,49.0,13.2,13.0,15.0,1,2,75.19,74.64,0.83,72.65


In [13]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

path_name = Path('../data/weather_data.json')
columns_names_to_drop = ['weather','weather_icon','sys.type']
columns_names_to_rename = {
        "base": "base",
        "visibility": "visibility",
        "dt": "datetime",
        "timezone": "timezone",
        "id": "city_id", 
        "name": "city_name",
        "cod": "code",
        "coord.lon": "longitude",
        "coord.lat": "latitude",
        "main.temp": "temperature",
        "main.feels_like": "feels_like",
        "main.temp_min": "temp_min",
        "main.temp_max": "temp_max",
        "main.pressure": "pressure",
        "main.humidity": "humidity",
        "main.sea_level": "sea_level",
        "main.grnd_level": "grnd_level",
        "wind.speed": "wind_speed",
        "wind.deg": "wind_deg",
        "wind.gust": "wind_gust",
        "clouds.all": "clouds", 
        "sys.type": "sys_type",                 
        "sys.id": "sys_id",                
        "sys.country": "country",                
        "sys.sunrise": "sunrise",                
        "sys.sunset": "sunset",
        # weather_id, weather_main, weather_description 
    }


def create_dataframe(path_name:str) -> pd.DataFrame:
    logging.info('Criando DataFrame do arquivo JSON...')
    path = path_name

    if not path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}')

    with open(path) as f:
        data = json.load(f)

    df = pd.json_normalize(data)
    logging.info(f'DataFrame criado com {len(df)} linha(s)...')
    return df

def normalize_weather_columns(df:pd.DataFrame) -> pd.DataFrame:
    df_weather = pd.json_normalize(df['weather'].apply(lambda x: x[0]))
    df_weather = df_weather.rename(columns={
        'id': 'weather_id',
        'main':'weather_main',
        'description':'weather_description',
        'icon':'weather_icon'
    })

    df = pd.concat([df, df_weather], axis=1)

    logging.info(f"\n Coluna 'weather' normalizada - {len(df.columns)} colunas")
    
    return df

def drop_columns(df:pd.DataFrame, columns_names:list[str]) -> pd.DataFrame:
    logging.info(f"\n Removendo colunas: {columns_names}")

    df = df.drop(columns=columns_names)
    logging.info(f"\n Colunas removidas - {len(df.columns)} colunas restantes")
    return df

def rename_columns(df: pd.DataFrame, columns_names:dict[str, str]) -> pd.DataFrame:
    logging.info(f"\n→ Renomeando {len(columns_names)} colunas...")
    df = df.rename(columns=columns_names)
    logging.info("✓ Colunas renomeadas")
    return df 

def normalize_datetime_columns(df: pd.DataFrame, columns_names:list[str]) -> pd.DataFrame:
    logging.info(f"\n→ Convertendo colunas para datetime: {columns_names}")
    for name in columns_names:
        df[name] = pd.to_datetime(df[name], unit='s', utc=True).dt.tz_convert('America/Sao_Paulo')

    logging.info(f"→ Colunas convertidas para datetime\n")
    return df

def data_transformations():
    print("\n Iniciando transformações")
    df = create_dataframe(path_name)
    df = normalize_weather_columns(df)
    df = drop_columns(df, columns_names_to_drop)
    df = rename_columns(df, columns_names_to_rename)
    df = normalize_datetime_columns(df, columns_to_normalize_datetime)
    logging.info("✓ Transformações concluídas\n")
    return df


create_dataframe(path_name)
# normalize_weather_columns(df)

2026-10-28 15:20:23,047 - root - INFO - Criando DataFrame do arquivo JSON...
2026-10-28 15:20:23,049 - root - INFO - DataFrame criado com 1 linha(s)...


,location.name,location.region,location.country,location.lat,location.lon,location.tz_id,location.localtime_epoch,location.localtime,current.last_updated_epoch,current.last_updated,...,current.air_quality.o3,current.air_quality.so2,current.air_quality.pm2_5,current.air_quality.pm10,current.air_quality.us-epa-index,current.air_quality.gb-defra-index,current.short_rad,current.diff_rad,current.dni,current.gti
0,Sao Paulo,Sao Paulo,Brazil,-23.5333,-46.6167,America/Sao_Paulo,1793190519,2026-10-28 09:28,1793189700,2026-10-28 09:15,...,49.0,13.2,13.0,15.0,1,2,75.19,74.64,0.83,72.65
